# 📘 부록: RunPod GPU에서 실습하기

이 단원부터 과정 끝까지의 실습(딥러닝 학습, LLM 파인튜닝, 모델 서빙)은 GPU에서 돌립니다. 개인 PC에는 GPU가 없거나 메모리가 부족한 경우가 많아, 필요한 만큼만 GPU를 빌려 쓰는 클라우드 서비스 RunPod를 씁니다. 이 부록은 RunPod에서 GPU 서버를 만들고, 실습 폴더를 올려 교안을 실행하고, 끝나면 비용이 더 나가지 않게 정리하는 방법을 정리합니다.

**이 부록의 흐름**
- RunPod의 용어(파드, 템플릿, 스토리지)와 요금이 나가는 방식을 봅니다.
- 실습 단계에 맞는 GPU를 고르는 기준을 정리합니다.
- 파드를 만들고 Jupyter Lab으로 접속해 실습 폴더를 올립니다.
- GPU가 잡혔는지 확인하고 교안을 실행합니다.
- 실습이 끝나면 파드를 정지하거나 종료해 비용을 막습니다.

## 1. RunPod 한눈에 보기

### 1-1. 파드와 템플릿

- **파드(Pod):** RunPod에서 빌리는 GPU 서버 한 대입니다. GPU 종류와 개수, 디스크 크기를 정해 만들고, 켜져 있는 시간만큼 요금이 나갑니다(밀리초 단위로 정산).
- **템플릿(Template):** 파드에 미리 설치해 둘 프로그램 묶음입니다. 이 과정은 PyTorch와 CUDA, Jupyter Lab이 깔린 공식 PyTorch 템플릿을 씁니다.

### 1-2. 저장 공간: 무엇이 남고 무엇이 지워지나

파드 만들기 화면의 **Storage** 칸에서 저장 공간 두 가지를 정합니다. 파드를 정지(Stop)하거나 종료(Terminate)할 때 남는 것이 다릅니다.

| 저장 공간 | 무엇인가 | 정지하면 | 종료하면 | 요금 |
|---|---|---|---|---|
| 컨테이너 디스크(Container disk) | 파드에 딸린 기본 디스크(기본 50GB). 올린 파일과 설치한 패키지가 모두 여기에 저장됩니다 | 지워짐 | 지워짐 | 파드가 켜져 있는 동안만 |
| 네트워크 볼륨(Persistent storage) | 계정에 따로 만들어 두는 디스크. 파드에 연결하면 `/workspace`에 보입니다 | 남음 | 남음 | 지우기 전까지 매달 |

- **이 단원은 네트워크 볼륨 없이 컨테이너 디스크만 씁니다.** 데이터가 작아 다시 올리기 쉽기 때문입니다. 대신 정지하는 순간 파일이 모두 지워지므로 실습 결과는 끝내기 전에 내려받습니다.
- **모델을 내려받는 파인튜닝 단원부터는 네트워크 볼륨을 씁니다.** 내려받은 모델이 볼륨에 남아 다음 수업에 다시 받지 않아도 됩니다. 볼륨은 만든 지역(region)에 묶이므로 그 지역에 있는 GPU만 고를 수 있습니다.
- **크레딧이 0이 되면 파드가 멈추고, 네트워크 볼륨이 없는 파드는 데이터를 되살릴 수 없습니다.**

### 1-3. 요금이 나가는 방식

- **실행 중:** GPU 요금과 컨테이너 디스크 요금이 나갑니다. 브라우저를 닫아도 파드가 켜져 있으면 계속 나갑니다.
- **정지 중:** GPU 요금은 멈춥니다. 볼륨 없이 만든 파드는 정지 중 요금이 없습니다.
- **종료 뒤:** 파드 요금은 더 나가지 않습니다. 네트워크 볼륨은 파드와 따로 남아 지울 때까지 요금이 나갑니다.
- **배포 조건:** 크레딧이 있어야 파드를 만들 수 있습니다. GPU 가격은 바뀌므로 파드를 만들 때 화면에 보이는 시간당 가격을 확인합니다.

## 2. GPU 고르기

GPU를 고를 때 가장 먼저 보는 것은 **GPU 메모리(VRAM)** 입니다. 모델과 학습 데이터가 GPU 메모리에 다 들어가야 학습이 돌고, 메모리가 모자라면 `CUDA out of memory` 오류로 멈춥니다. 메모리가 충분하면 그다음은 세대(같은 일을 얼마나 빨리 하는가)와 시간당 가격을 봅니다.

### 2-1. 메모리 어림 계산

- **모델 불러오기:** 16비트로 불러온 모델은 파라미터 10억 개마다 약 2GB를 씁니다. 8B 모델이면 약 16GB입니다.
- **학습:** 기울기와 옵티마이저가 기억하는 값(AdamW는 파라미터마다 두 개)까지 더해져 불러오기만 할 때보다 몇 배 더 씁니다. 그래서 큰 모델은 LoRA·4비트 QLoRA처럼 학습할 값을 줄이는 방법을 씁니다.
- **배치 크기:** 배치를 키우면 한 번에 GPU에 올리는 데이터가 늘어 메모리를 더 씁니다. 메모리가 모자라면 배치부터 줄입니다.

### 2-2. RunPod의 GPU별 특징

| 메모리 | GPU | 시간당 가격(대략) | 학습 관점의 특징 |
|---|---|---|---|
| 16GB | RTX 2000 Ada·RTX A4000 | 0.25달러 | 가장 저렴합니다. 느리지만 작은 신경망·PyTorch 연습에는 충분합니다 |
| 20GB | RTX A4500·RTX 4000 Ada | 0.25~0.3달러 | 16GB와 값이 비슷하고 메모리가 조금 더 큽니다 |
| 24GB | RTX A5000 | 0.27달러 | 24GB 중 가장 저렴합니다. 7B 모델 LoRA·QLoRA 연습의 기본 선택입니다 |
| 24GB | RTX 3090·RTX 4090 | 0.5~0.9달러 | 게이밍용 고성능 카드입니다. 같은 24GB에서 학습이 가장 빠르고, 4090은 3090보다 학습이 대체로 1.5배 안팎(모델에 따라 1.2~1.9배) 빠릅니다 |
| 24GB | L4 | 0.6달러 | 전력이 낮은 추론(서빙)용 카드입니다. 메모리 대역폭이 좁아 학습은 A5000보다 느립니다 |
| 32GB | RTX 5090 | 1.2달러 | 최신 게이밍 카드입니다. 24GB로 조금 모자랄 때 고릅니다 |
| 48GB | A40·RTX A6000 | 0.6달러 | 48GB 중 가장 저렴합니다. 8B 모델을 16비트로 띄우는 서빙 실습에 맞습니다 |
| 48GB | L40S·RTX 6000 Ada | 1~1.1달러 | A40보다 한 세대 새 카드라 학습·추론이 더 빠릅니다 |
| 80GB | A100 | 1.8달러 | 데이터센터용입니다. 빠른 HBM 메모리를 써서 큰 모델 학습이 빠릅니다 |
| 80GB 이상 | H100·H200·B200 | 3~8달러 | 최상위 데이터센터용입니다. 대형 모델 사전학습·다중 GPU 학습에 쓰며 이 과정에는 필요하지 않습니다 |

- **세대:** Ampere(RTX A·3090·A40·A100)에서 Ada(RTX Ada·4090·L4·L40S)·Hopper(H100·H200)를 거쳐 Blackwell(5090·B200)로 이어집니다. Ada에는 그래픽 카드(RTX 4090·RTX Ada)와 데이터센터 카드(L4·L40S)가 함께 있고, Hopper는 데이터센터 전용입니다. 같은 계열이면 새 세대가 학습이 빠르지만, 계열이 다르면 옛 세대가 빠를 수도 있습니다(A100이 L4보다 훨씬 빠릅니다).
- **정밀도:** 이 과정이 쓰는 16비트(bf16) 학습은 Ampere부터 모두 됩니다. Ada·Hopper부터 FP8, Blackwell부터 FP4 같은 더 낮은 정밀도 계산을 하드웨어가 직접 지원합니다.
- **이름의 뜻:** 이름에 SXM이 붙은 카드는 GPU끼리 빠르게 연결돼 여러 장으로 학습할 때 유리하고, PCIe는 한 장으로 쓸 때 충분합니다. MIG는 큰 GPU 한 장을 여러 조각으로 나눈 것이라 이름에 적힌 만큼의 메모리와 그에 맞는 연산 능력만 씁니다.
- **AMD GPU:** MI300X 같은 AMD GPU는 CUDA가 아니라 ROCm을 씁니다. 이 과정의 코드와 PyTorch 템플릿은 NVIDIA(CUDA) 기준이므로 NVIDIA GPU를 고릅니다.
- **가격과 자리:** 위 가격은 파드 만들기 화면에 보이는 대략적인 값이고 수시로 바뀝니다. 싼 GPU는 빈 자리가 없을 때가 많습니다(카드에 **Out of capacity** 표시). GPU 목록의 **Available** 탭에는 지금 빌릴 수 있는 GPU만 보입니다.

### 2-3. 이 과정의 단계별 권장 GPU

| 단계 | 실습 | 권장 GPU 메모리 |
|---|---|---|
| 딥러닝 학습 원리·PyTorch·트랜스포머 | 작은 텐서와 신경망 | 16GB(RTX 2000 Ada·RTX A4000) |
| 파인튜닝(SFT·LoRA·QLoRA·DPO)·임베딩 파인튜닝 | 1.7B~8B 모델 학습(8B는 4비트 QLoRA) | 24GB(RTX A5000·RTX 3090·RTX 4090) |
| 서빙(vLLM)·벤치마크·배포 | 8B 모델을 16비트로 띄워 요청 처리 | 48GB(A40·RTX A6000·L40S) |

이 단원은 데이터가 작아 어떤 GPU에서도 몇 초 안에 끝나므로 가장 저렴한 16GB GPU(RTX 2000 Ada·RTX A4000)면 충분합니다. 이 부록의 화면은 그날 싼 GPU에 빈 자리가 없어 A40으로 찍었으므로, GPU 이름과 가격은 화면과 다르게 보일 수 있습니다. 고르려는 GPU가 목록에 없으면 지금 빈 자리가 없다는 뜻이므로 같은 메모리의 다른 GPU를 고릅니다.

## 3. 파드 만들기

### 3-1. 크레딧 충전

RunPod는 미리 충전한 크레딧에서 요금을 빼 가는 방식입니다. 콘솔 왼쪽 메뉴의 **Billing**에서 크레딧을 충전합니다. 잔액이 줄어들 때 메일로 알려 주는 설정도 같은 화면에 있습니다.

### 3-2. 파드 만들기 화면 열기

콘솔 왼쪽 메뉴의 **Pods**를 누르면 **Deploy a Pod** 화면이 열립니다(이미 파드가 있으면 파드 목록 위의 **Deploy** 버튼을 누릅니다). 한 화면에서 위에서부터 템플릿, 이름, 지역, GPU, 저장 공간을 정하고, 오른쪽 요약 칸에서 배포합니다.

<img src="images/runpod_01_pods.png" width="1050" alt="RunPod 콘솔의 Deploy a Pod 화면. 왼쪽 메뉴의 Pods와 화면 제목 Deploy a Pod에 빨간 사각형이 그려져 있고, 아래에 PyTorch 템플릿과 Pod name 칸이 보인다.">

### 3-3. 템플릿과 이름

- **Workload:** 공식 **PyTorch** 템플릿이 기본으로 골라져 있습니다. 다른 템플릿이 보이면 검색 칸에서 PyTorch를 찾아 고릅니다.
- **Pod name:** 파드 이름입니다. `day65-practice`처럼 알아보기 쉽게 적습니다.
- **Set overrides:** 누르면 템플릿 설정을 바꾸는 **Edit template overrides** 창이 열립니다. **Start Jupyter notebook**이 체크돼 있어야 Jupyter Lab으로 접속할 수 있으니 확인만 하고 오른쪽 위 X로 닫습니다.

<img src="images/runpod_02_template.png" width="1050" alt="Set overrides를 누르면 열리는 Edit template overrides 창. 컨테이너 이미지 runpod/pytorch와 Start Jupyter notebook 체크 상자에 빨간 사각형이 그려져 있다.">

### 3-4. GPU 고르기

- **Region**은 **Any region**으로 둡니다. 자리가 있는 모든 지역의 GPU가 보입니다.
- **Compute**의 **GPU** 탭에서 **Available**(지금 빌릴 수 있는 GPU)을 보고, 2절 기준에 맞는 GPU 카드를 누릅니다. 목록은 메모리가 큰 순서라 16GB 카드는 아래쪽에 있습니다. **Search GPUs** 칸에 `RTX 2000`이나 `A4000`을 입력하면 바로 찾을 수 있습니다.
- 그 아래 **GPU count**는 1, **Instance pricing**은 쓴 만큼 내는 **On-Demand**로 둡니다(**Reserved**는 기간을 미리 약정하는 요금입니다).

<img src="images/runpod_03_gpu.png" width="1050" alt="Compute 칸의 GPU 목록. Available 탭과 고른 GPU 카드(A40, 48GB, 시간당 0.59달러)에 빨간 사각형이 그려져 있다.">

### 3-5. 저장 공간과 배포

- **Container disk:** 기본값 50GB로 둡니다.
- **Persistent storage:** 이 단원은 비워 둡니다. 요약 칸에 `Nothing mounted at the template's path`(템플릿이 권하는 `/workspace` 볼륨이 없다는 안내)가 떠도 그대로 배포합니다.
- 오른쪽 요약 칸의 **Total / hour**(GPU와 디스크를 합한 시간당 요금)를 확인하고 **Deploy Pod**를 누릅니다. 버튼이 **Deploy when available**로 바뀌어 있으면 고른 GPU에 빈 자리가 없다는 뜻이므로 누르지 말고 다른 GPU를 고릅니다.

<img src="images/runpod_04_summary.png" width="1050" alt="배포 직전 오른쪽 요약 칸. Total / hour 0.60달러와 Deploy Pod 버튼에 빨간 사각형이 그려져 있다.">

## 4. 접속하고 실습 폴더 올리기

### 4-1. Jupyter Lab 열기

배포하면 Pods 화면 오른쪽에 파드 상세 칸이 열립니다. **Connect** 탭의 **HTTP services**에서 **Port 8888 → Jupyter Lab**이 **Ready**가 되면 링크를 눌러 Jupyter Lab을 엽니다. 파드가 시작되는 데 1~2분 걸립니다.

<img src="images/runpod_05_connect.png" width="1050" alt="실행 중인 파드의 상세 칸. Connect 탭, Jupyter Lab 링크, 준비 완료 표시 Ready에 빨간 사각형이 그려져 있다.">

### 4-2. 실습 폴더 올리기와 패키지 설치

- **폴더 올리기:** 폴더 안의 파일이 아니라 `day65_딥러닝_학습원리` 폴더 자체를 zip으로 묶습니다. 그래야 `/workspace/day65_딥러닝_학습원리`가 생깁니다. Jupyter Lab 왼쪽 파일 목록에서 `workspace` 폴더로 들어가 zip 파일을 끌어다 놓습니다. 실습 자료가 GitHub 저장소에 있으면 터미널에서 `git clone`으로 받아도 됩니다.
- **터미널 열기:** Jupyter Lab의 **File → New → Terminal**입니다.
- **압축 풀기와 패키지 설치:** 압축은 파이썬 기본 기능으로 풉니다. PyTorch는 템플릿에 들어 있으므로 나머지만 설치하고, 그래프의 한글이 네모로 나오지 않게 나눔고딕 글꼴도 설치합니다.

```bash
cd /workspace
python -m zipfile -e day65_딥러닝_학습원리.zip .
pip install pandas matplotlib
apt-get update && apt-get install -y fonts-nanum
rm -rf ~/.cache/matplotlib
```

압축을 푼 뒤 `ls day65_딥러닝_학습원리/data`로 `본실습`·`따라하기`·`과제01`·`과제02` 폴더 이름이 바르게 보이는지 확인합니다. 한글이 깨져 보이면 강사가 배포한 zip을 받아 다시 올립니다.

볼륨 없이 만든 파드는 정지하면 올린 폴더와 설치한 패키지가 모두 지워집니다. 다시 켜면 폴더 올리기부터 다시 합니다.

## 5. GPU 확인하고 교안 실행하기

### 5-1. GPU가 잡혔는지 확인

파드의 Jupyter Lab에서 새 노트북을 만들고, 아래 두 코드를 각각 셀에 붙여 실행합니다. `nvidia-smi`는 GPU 이름, 메모리, 지금 쓰고 있는 메모리를 보여 주는 명령이고, 셀 맨 앞의 `!`는 셀 안에서 터미널 명령을 실행하라는 표시입니다.

```python
# GPU 이름과 메모리, 드라이버가 지원하는 CUDA 버전을 확인합니다
!nvidia-smi
```

```python
import torch

# PyTorch가 GPU를 쓸 수 있는지, 어떤 GPU인지 확인합니다
print("PyTorch 버전:", torch.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    print("GPU 이름:", torch.cuda.get_device_name(0), "| GPU 메모리(GB):", round(total_gb, 1))
```

- **CUDA 버전이 달라 보여도 됩니다:** `nvidia-smi`의 CUDA Version은 드라이버가 지원하는 가장 높은 버전입니다. PyTorch 버전의 `cu128`(CUDA 12.8용)보다 높으면 그대로 씁니다.
- **메모리가 조금 작게 보여도 됩니다:** 48GB 카드가 44.4로 보이는 것은 1GB를 1024MB로 나눠 계산했기 때문입니다.

<img src="images/runpod_06_gpu_check.png" width="1050" alt="파드의 Jupyter Lab에서 nvidia-smi와 torch.cuda 확인 셀을 실행한 화면. nvidia-smi의 GPU 이름(NVIDIA A40)과 전체 메모리(46068MiB), 출력 'GPU 사용 가능: True'에 빨간 사각형이 그려져 있다.">

### 5-2. 교안 실행

`/workspace/day65_딥러닝_학습원리`에서 교안 노트북을 열고 커널로 **Python 3 (ipykernel)** 을 고른 뒤 위에서부터 실행합니다. 교안의 준비 셀이 `계산 장치: cuda | (GPU 이름)`을 출력하면 GPU에서 계산하고 있는 것입니다. `cpu`가 나오면 파드를 GPU 없이 만들었거나 PyTorch가 CUDA를 찾지 못한 것이니 GPU 확인 셀부터 다시 실행합니다.

<img src="images/runpod_07_lesson_cuda.png" width="1050" alt="파드의 Jupyter Lab에서 교안 01 준비 셀을 실행한 화면. 출력 '계산 장치: cuda'와 GPU 이름에 빨간 사각형이 그려져 있다.">

## 6. 끝내기: 정지(Stop)와 종료(Terminate)

실습이 끝나면 반드시 파드를 정지하거나 종료합니다. 브라우저 탭을 닫는 것만으로는 파드가 꺼지지 않아 요금이 계속 나갑니다.

| 버튼 | 하는 일 | 언제 쓰나 |
|---|---|---|
| **Stop**(정지) | GPU를 반납하고 파드 설정은 남깁니다. 볼륨 없이 만든 파드는 컨테이너 디스크가 지워져 파일이 모두 사라집니다 | 같은 설정으로 다시 켤 때 |
| **Terminate**(종료) | 파드를 지웁니다. 되돌릴 수 없습니다 | 결과를 내려받고 더 쓰지 않을 때 |

**끝내는 순서**
1. 결과를 내려받습니다. 터미널에서 `cd /workspace && python -m zipfile -c day65_결과.zip day65_딥러닝_학습원리`로 묶은 뒤, 파일 목록에서 zip을 오른쪽 클릭해 **Download**를 누릅니다.
2. 파드 상세 칸의 **Stop**을 누르고 확인 창의 **Stop Pod**를 누릅니다. 볼륨이 없으면 창에 `ALL DATA will be lost!`(모든 데이터가 지워진다)가 뜨고, 정지 중 요금(Idle disk cost)은 0입니다.
3. 정지되면 버튼이 **Start**, **Terminate**, **Request support**로 바뀝니다. **Terminate**를 누르고 확인 창의 **Terminate Pod**를 누릅니다. 실행 중인 파드에는 Terminate 버튼이 없어서 먼저 정지해야 합니다.
4. Pods 화면에 파드가 남아 있지 않은지 확인합니다.

<img src="images/runpod_08_stop.png" width="1050" alt="정지 확인 창. 볼륨이 없어 모든 데이터가 지워진다는 경고와 Stop Pod 버튼에 빨간 사각형이 그려져 있다.">

<img src="images/runpod_09_terminate.png" width="1050" alt="정지된 파드의 상세 칸. 정지한 뒤 나타난 Terminate 버튼에 빨간 사각형이 그려져 있다.">

- **지출 확인:** 콘솔의 **Billing**에서 날짜별·파드별 사용 금액을 봅니다. 네트워크 볼륨은 파드를 지워도 남아 매달 요금이 나가므로, **Storage** 메뉴에서 안 쓰는 볼륨이 없는지 함께 봅니다.

## 7. 자주 막히는 곳

- **`CUDA out of memory`:** 모델이나 배치가 GPU 메모리에 다 들어가지 않습니다. 배치 크기를 줄이거나 메모리가 더 큰 GPU로 파드를 새로 만듭니다.
- **배포 버튼이 Deploy when available로 바뀜:** 고른 GPU의 자리가 없다는 뜻입니다(카드에 Out of capacity). **Available** 탭에서 다른 GPU를 고릅니다. 버튼이 아예 눌리지 않으면 크레딧이 부족한지 봅니다.
- **Jupyter Lab이 열리지 않음:** 파드가 아직 시작 중일 수 있습니다. Connect 탭에서 Jupyter Lab이 Ready인지 확인합니다. 파드를 만들 때 Set overrides의 Start Jupyter notebook이 꺼져 있었다면, 파드를 종료하고 체크한 채로 새로 만듭니다.
- **정지 후 다시 켜니 폴더와 패키지가 없음:** 볼륨 없이 만든 파드는 정지하면 컨테이너 디스크가 지워집니다. 폴더 올리기와 설치 명령을 다시 실행합니다.
- **정지했다 다시 켜니 GPU가 0개:** 원래 서버의 GPU를 다른 사람이 쓰고 있으면 GPU가 0개로 뜰 수 있습니다. 파드를 종료하고 새로 만듭니다.
- **그래프의 한글이 네모로 나옴:** 나눔고딕을 설치하고 `~/.cache/matplotlib`을 지운 뒤 커널을 다시 시작합니다.

## 8. 핵심 정리

- **파드는 켜져 있는 동안 요금이 나갑니다.** 실습이 끝나면 정지한 뒤 종료합니다.
- **볼륨 없이 만든 파드는 정지하면 파일이 모두 지워집니다.** 실습 결과는 끝내기 전에 내려받습니다.
- **GPU는 메모리부터 고릅니다.** 이 단원은 16GB, 파인튜닝은 24GB, 서빙은 48GB면 충분합니다.
- **GPU 확인은 두 줄입니다.** `nvidia-smi`와 `torch.cuda.is_available()`로 확인하고, 교안 준비 셀의 `계산 장치: cuda`를 봅니다.